In [2]:
import pandas as pd
import numpy  as np
import torch
from transformers import BertTokenizer, BertModel, BertConfig
from transformers import BertForSequenceClassification, Trainer, TrainingArguments ,TrainerCallback
from sklearn.model_selection import train_test_split
import re

2025-05-03 18:20:30.337834: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746296430.536852      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746296430.591253      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
model_path = "/kaggle/input/bert-base-binary/Bert-base-binary"
tokenizer = BertTokenizer.from_pretrained(model_path)

In [4]:
from datasets import load_dataset

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    label = int(example["hate"])
    enc["labels"] = [1.0, 0.0] if label == 0 else [0.0, 1.0]

    return enc

tokenized_ds = ds.map(preprocess)

README.md:   0%|          | 0.00/8.28k [00:00<?, ?B/s]

multilabel_train.csv:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

multilabel_test.csv:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/34934 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8734 [00:00<?, ? examples/s]

Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [5]:
import torch.nn as nn

class BertWithCNN(nn.Module):
    def __init__(self, model_path, num_labels=2, freeze_bert=True):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_path)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=self.bert.config.hidden_size, out_channels=128, kernel_size=k)
            for k in [2, 3, 4]
        ])
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(128 * len(self.convs), num_labels)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state        # [B, L, H]
        x = x.permute(0, 2, 1)               # [B, H, L]

        x = [torch.relu(conv(x)) for conv in self.convs]
        x = [torch.max(conv, dim=2)[0] for conv in x]
        x = torch.cat(x, dim=1)              # [B, 128 * 3]

        x = self.dropout(x)
        logits = self.classifier(x)

        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}


In [6]:
def unfreeze_layers(model, n_layers_to_unfreeze):
    encoder = model.bert.encoder
    for i in range(11, 11 - n_layers_to_unfreeze, -1):
        for param in encoder.layer[i].parameters():
            param.requires_grad = True

In [7]:
class GradualUnfreezingCallback(TrainerCallback):
    def __init__(self, model, unfreeze_every_n_epochs=1, total_layers=12):
        self.model = model
        self.unfreeze_every_n_epochs = unfreeze_every_n_epochs
        self.total_layers = total_layers
        self.unfrozen = 0

    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch is not None and int(state.epoch) % self.unfreeze_every_n_epochs == 0:
            if self.unfrozen < self.total_layers:
                self.unfrozen += 1
                unfreeze_layers(self.model, self.unfrozen)
                print(f"✅ Unfroze top {self.unfrozen} BERT layer(s).")

In [7]:
model = BertWithCNN(model_path, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',  # Directory to save the model checkpoints    eval_strategy="epoch",  # Evaluate after each epoch
    logging_strategy="no",  # Disables logging
    eval_strategy="epoch",
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=7,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)

In [10]:
from sklearn.metrics import accuracy_score
def compute_metrics(pred):
    logits = pred.predictions
    labels = pred.label_ids
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = (np.array(labels) > 0.5).astype(int)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    callbacks=[GradualUnfreezingCallback(model)],
    compute_metrics=compute_metrics
)

/tmp/ipykernel_31/1869630783.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.204312,0.918136
2,No log,0.204335,0.918480
3,No log,0.203946,0.917793
4,No log,0.206179,0.919510
5,No log,0.202795,0.919166
6,No log,0.204938,0.919281
7,No log,0.204327,0.918823


✅ Unfroze top 1 BERT layer(s).
✅ Unfroze top 2 BERT layer(s).
✅ Unfroze top 3 BERT layer(s).
✅ Unfroze top 4 BERT layer(s).
✅ Unfroze top 5 BERT layer(s).
✅ Unfroze top 6 BERT layer(s).
✅ Unfroze top 7 BERT layer(s).


TrainOutput(global_step=15288, training_loss=0.21378657382424124, metrics={'train_runtime': 1688.4195, 'train_samples_per_second': 144.832, 'train_steps_per_second': 9.055, 'total_flos': 0.0, 'train_loss': 0.21378657382424124, 'epoch': 7.0})

In [12]:
!mkdir ./CNN_grad_unfreeze_bert
torch.save(model, './CNN_grad_unfreeze_bert/CNN_grad_unfreeze_bert.pth')
tokenizer.save_pretrained("./CNN_grad_unfreeze_bert")

('./CNN_grad_unfreeze_bert/tokenizer_config.json',
 './CNN_grad_unfreeze_bert/special_tokens_map.json',
 './CNN_grad_unfreeze_bert/vocab.txt',
 './CNN_grad_unfreeze_bert/added_tokens.json')

In [13]:
model = BertWithCNN(model_path, num_labels=2)

training_args2 = TrainingArguments(
    output_dir='./results2',  # Directory to save the model checkpoints    eval_strategy="epoch",  # Evaluate after each epoch
    logging_strategy="no",  # Disables logging
    eval_strategy="epoch",
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=5,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs2',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)


trainer_nn = Trainer(
    model=model,
    args=training_args2,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_31/4184583580.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_nn = Trainer(


In [14]:
trainer_nn.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.204396,0.918022
2,No log,0.203605,0.919281
3,No log,0.202566,0.918937
4,No log,0.202579,0.918937
5,No log,0.201476,0.918823


TrainOutput(global_step=10920, training_loss=0.2140903794285142, metrics={'train_runtime': 855.8777, 'train_samples_per_second': 204.083, 'train_steps_per_second': 12.759, 'total_flos': 0.0, 'train_loss': 0.2140903794285142, 'epoch': 5.0})

In [15]:
!mkdir ./CNN_model
torch.save(model, './CNN_model/CNN_model.pth')
tokenizer.save_pretrained("./CNN_model")

('./CNN_model/tokenizer_config.json',
 './CNN_model/special_tokens_map.json',
 './CNN_model/vocab.txt',
 './CNN_model/added_tokens.json')

In [16]:
model_path = "/kaggle/input/bert-large/Bert-large-binary"
model = BertWithCNN(model_path, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results3',  # Directory to save the model checkpoints    eval_strategy="epoch",  # Evaluate after each epoch
    logging_strategy="no",  # Disables logging
    eval_strategy="epoch",
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=5,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)

In [17]:
trainer3 = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    callbacks=[GradualUnfreezingCallback(model)],
    compute_metrics=compute_metrics
)

/tmp/ipykernel_31/1911502760.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer3 = Trainer(


In [18]:
trainer3.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.137799,0.949508
2,No log,0.139701,0.950424
3,No log,0.138720,0.949279
4,No log,0.141304,0.950882
5,No log,0.139775,0.949393


✅ Unfroze top 1 BERT layer(s).
✅ Unfroze top 2 BERT layer(s).
✅ Unfroze top 3 BERT layer(s).
✅ Unfroze top 4 BERT layer(s).
✅ Unfroze top 5 BERT layer(s).


TrainOutput(global_step=10920, training_loss=0.15577620621565935, metrics={'train_runtime': 4510.5525, 'train_samples_per_second': 38.725, 'train_steps_per_second': 2.421, 'total_flos': 0.0, 'train_loss': 0.15577620621565935, 'epoch': 5.0})

In [20]:
!mkdir ./CNN_grad_unfreeze_bertl
torch.save(model, './CNN_grad_unfreeze_bertl/CNN_grad_unfreeze_bertl.pth')
tokenizer.save_pretrained("./CNN_grad_unfreeze_bertl")

('./CNN_grad_unfreeze_bertl/tokenizer_config.json',
 './CNN_grad_unfreeze_bertl/special_tokens_map.json',
 './CNN_grad_unfreeze_bertl/vocab.txt',
 './CNN_grad_unfreeze_bertl/added_tokens.json')